# Notebook 03 — Anomaly Detection

**Question:** trained only on data assumed healthy, how early does the detector flag the
bearing that eventually fails?

- Fit on the **first 60% of each run**, taken as healthy
- Score the whole series
- Measure how far before the end the first flag appears

**No labels reach the model.** The failure mode is known from the IMS documentation, but it
is used only to decide which bearing to look at afterwards.

---

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from nasa_bearing_anomaly.config import FIGURES_DIR, TEST_CONFIG
from nasa_bearing_anomaly.detection import run_pipeline, select_features

# Importing plotting applies the project's dark-industrial rcParams as a module-level
# side effect, and STYLE is the palette those figures are drawn from. Without this
# import the two figures below render on the matplotlib default -- white ground,
# default grid -- while every other figure in the repository is dark, so the site
# would show one analysis in two visual languages.
from nasa_bearing_anomaly.plotting import STYLE

print("Imports OK")

## 1. Why Isolation Forest?

**Core idea:** Normal data points are hard to isolate — they live in dense clusters and require many splits to separate. Anomalous points are easy to isolate — they're in sparse regions and need fewer splits.

The algorithm builds random trees and measures the **average path length** to isolate each point. Short path → anomaly.

**Advantages for this problem:**
- No labels required
- Scales well to many features
- Robust to the different failure signatures in Tests 1/2/3
- Interpretable anomaly score

## 2. Run Detection: All Three Tests

In [ ]:
results = {}

for test_id in [1, 2, 3]:
    print(f"\n{'=' * 60}")
    print(
        f"Test {test_id}: {TEST_CONFIG[test_id]['failed_bearing']} — {TEST_CONFIG[test_id]['failure_mode']}"
    )
    print(f"{'=' * 60}")
    results[test_id] = run_pipeline(test_id, method="isolation_forest")

## 3. Results Summary

In [ ]:
print("\n" + "═" * 70)
print("ANOMALY DETECTION RESULTS SUMMARY")
print("═" * 70)
print(
    f"{'Test':<6} {'Failed Bearing':<15} {'Files':<8} {'Anomalies':<11} "
    f"{'First flag':<13} {'Raw Lead'}"
)
print("─" * 70)

for test_id, df in results.items():
    config = TEST_CONFIG[test_id]
    n = len(df)
    n_anom = df["is_anomaly"].sum()
    first_anom = df[df["is_anomaly"]].index.min() if n_anom > 0 else n

    # Lead time measured from the acquisition timestamps, not from an assumed
    # sampling interval. The timestamps parse correctly as of 2026-08-12.
    stamps = pd.to_datetime(df["timestamp"])
    lead_hours = (
        (stamps.iloc[-1] - stamps.loc[first_anom]).total_seconds() / 3600 if n_anom > 0 else 0.0
    )

    print(
        f"{test_id:<6} {config['failed_bearing']:<15} {n:<8} "
        f"{n_anom:<11} {first_anom:<13} {lead_hours:>6.1f} h"
    )

print("═" * 70)
print()
print("These are RAW lead times, and they are NOT a result. They are printed to show")
print("why the obvious metric fails, which is a measurement rather than a prediction:")
print("every run flags file 0, so 'raw lead' is just the length of the run.")
print()
print("Three things are missing:")
print("  1. The trigger is a single flagged file. contamination labels that fraction of")
print("     the model's own training window anomalous by construction, so the first flag")
print("     lands at or near 0 no matter how healthy the bearing is. A sustained rule --")
print("     k of the last m files flagged -- replaces it.")
print("  2. No false-alarm rate. A lead time without its false-alarm cost is not a")
print("     finding, and the rate has to be calibrated on held-out healthy data.")
print("  3. The post-shutdown tail has to be excluded, and it is a DIFFERENT LENGTH in")
print("     each run -- 0 files for Test 1, 2 for Test 2, 1 for Test 3. Test 1 never went")
print("     quiet and ends on its RMS peak, so dropping 'the last file' there would")
print("     discard the very failure being measured. See data/README.md.")
print()
print("All three are implemented in business.py. Notebook 04 reports the result:")
print("79.5 / 53.7 / 57.0 h, against 0 sustained alarms in 323 / 147 / 949 held-out")
print("healthy files. No euro figure follows from the raw numbers above.")

## 4. Visualization: All Tests

Raw detector output, one panel pair per run. The red markers are `is_anomaly`
straight from the Isolation Forest, and they are deliberately **not** an alert:
`contamination` labels that fraction of the model's own training window anomalous
by construction, so flagged files appear throughout the healthy region of every
run and the first of them lands at or near file 0.

Turning this into something a maintenance team could act on needs a sustained-alert
rule and a score threshold calibrated against held-out healthy data. That is
notebook 04 and `business.py`. Nothing in these panels is a lead time.

In [ ]:
for test_id, df in results.items():
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    rms_col = f"{failed}_ch1_rms"
    score_col = "anomaly_score"

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(
        f"Test {test_id} — {failed} | {config['failure_mode']}", fontsize=12, fontweight="bold"
    )

    # RMS. No first-alert line: it would sit at file 0 on every run, which is an
    # artefact of contamination rather than a detection. The sustained alarm is
    # derived in notebook 04 and marked there.
    if rms_col in df.columns:
        ax1.plot(df.index, df[rms_col], color=STYLE["accent_color"], linewidth=0.8, label="RMS")
        if "is_anomaly" in df.columns:
            anom_mask = df["is_anomaly"]
            ax1.scatter(
                df.index[anom_mask],
                df[rms_col][anom_mask],
                color=STYLE["anomaly_color"],
                s=8,
                zorder=5,
                label="Flagged file (raw)",
            )

    ax1.set_ylabel("RMS (g)")
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.4)

    # Anomaly score
    if score_col in df.columns:
        s = df[score_col]
        s_norm = (s - s.min()) / (s.max() - s.min() + 1e-12)
        ax2.plot(df.index, s_norm, color=STYLE["warn_color"], linewidth=0.7)
        if "is_anomaly" in df.columns:
            ax2.fill_between(
                df.index,
                s_norm,
                where=df["is_anomaly"],
                color=STYLE["anomaly_color"],
                alpha=0.4,
                label="Flagged",
            )
        ax2.legend(fontsize=8)

    ax2.set_ylabel("Anomaly Score")
    ax2.set_xlabel("File Index")
    ax2.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / f"detection_test{test_id}.png",
        dpi=150,
        bbox_inches="tight",
        facecolor=STYLE["bg_color"],
    )
    plt.show()

## 5. PCA Feature Space Visualization

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("PCA Feature Space — Normal (green) vs Anomaly (red)", fontsize=12, fontweight="bold")

for ax, (test_id, df) in zip(axes, results.items(), strict=True):
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]

    feature_cols = select_features(df, bearing_prefix=failed)
    if len(feature_cols) < 2:
        feature_cols = select_features(df)

    X = df[feature_cols].fillna(0).to_numpy()
    X_scaled = StandardScaler().fit_transform(X)
    X_2d = PCA(n_components=2).fit_transform(X_scaled)

    colors = [
        STYLE["anomaly_color"] if a else STYLE["healthy_color"]
        for a in df.get("is_anomaly", [False] * len(df))
    ]
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=colors, s=10, alpha=0.6)
    ax.set_title(
        f"Test {test_id} — {failed}\n{config['failure_mode']}", fontsize=10, fontweight="bold"
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.3)

legend_elems = [
    Patch(facecolor=STYLE["healthy_color"], label="Normal"),
    Patch(facecolor=STYLE["anomaly_color"], label="Anomaly"),
]
fig.legend(
    handles=legend_elems, loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05)
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "pca_all_tests.png",
    dpi=150,
    bbox_inches="tight",
    facecolor=STYLE["bg_color"],
)
plt.show()

## 6. (Optional) Autoencoder for Test 3

Test 3 has a gradual multi-phase degradation that benefits from the autoencoder's deeper representation learning.

In [ ]:
from nasa_bearing_anomaly.detection import TORCH_AVAILABLE

if TORCH_AVAILABLE:
    print("Running Autoencoder on Test 3...")
    ae_result = run_pipeline(3, method="autoencoder")

    # Compare the two detectors on the same diagnostic. Both land at 0 for the same
    # structural reason, which is the point: swapping the model does not fix a metric
    # that was never measuring detection. business.py is what fixes it, and it handles
    # the autoencoder's inverted sign convention.
    if_first = (
        results[3][results[3]["is_anomaly"]].index.min() if results[3]["is_anomaly"].any() else None
    )
    ae_first = (
        ae_result[ae_result["is_anomaly"]].index.min() if ae_result["is_anomaly"].any() else None
    )

    print("\nTest 3, first flagged file (a diagnostic, not a lead time):")
    print(f"  Isolation Forest: {if_first}")
    print(f"  Autoencoder:      {ae_first}")
    print("\nNo autoencoder result is published; it did not earn a place over the")
    print("Isolation Forest on this data.")
else:
    print("PyTorch not installed. Install the optional extra with: pip install -e '.[deep]'")

---
## Summary

The Isolation Forest separates the failed bearing from its healthy history on every run.
What it does *not* do is hand over a lead time: the numbers printed above trigger on the
first flagged file, and that file is index 0 on all three runs.

That is not a tuning problem. `IsolationForest(contamination=c)` labels that fraction of
its **training** window anomalous by construction, and the training window is the first
60% — the part assumed healthy. Measured: 5.03% / 5.08% / 8.01% of training files flagged
against 0.05 / 0.05 / 0.08 configured. So the raw "lead time" is just the run length, and
no choice of model changes that — the autoencoder above lands at 0 as well.

Three things convert it into something usable, all of them in `business.py`:

- a **sustained-alert rule**, k of the last m files flagged, with k and m chosen on a
  healthy window that is disjoint from the one the rate is reported on
- a **score threshold calibrated on held-out healthy data**, never `is_anomaly`
- the **post-shutdown tail excluded**, measured per run rather than assumed — 0 files for
  Test 1, 2 for Test 2, 1 for Test 3

Applied, those give **79.5 / 53.7 / 57.0 hours** of warning against **0 sustained alarms**
in 323 / 147 / 949 held-out healthy files. Notebook 04 computes and reports them.

**Next:** Results visualization & business report → Notebook 04